# FD-DRAT on LIBERO10 (Kaggle)

Pipeline:
1. Setup environment
2. Download LIBERO10 dataset
3. Train OAT tokenizer (~2-3h on T4)
4. Train FD-DRAT policy

**Requirements:** Enable GPU (T4 x2 or P100), add `WANDB_API_KEY` to Kaggle Secrets.

## Cell 1 — GPU check & W&B secret

In [ ]:
import os
import subprocess

# Verify GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())

# W&B API key from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
    print('W&B API key loaded from Kaggle Secrets')
except Exception as e:
    print(f'W&B secret not found ({e}), running in offline mode')
    os.environ['WANDB_MODE'] = 'offline'

WORKDIR = '/kaggle/working'
os.chdir(WORKDIR)
print('Working dir:', os.getcwd())

## Cell 2 — Clone repos

In [ ]:
%%bash
set -e

# OAT (with LIBERO submodule)
if [ ! -d oat ]; then
    git clone --recurse-submodules https://github.com/Chaoqi-LIU/oat.git oat
else
    echo 'oat/ already exists, skipping clone'
fi

# HNet (not used in training but needed by sys.path in run.py)
if [ ! -d hnet ]; then
    git clone https://github.com/goombalab/hnet.git hnet
else
    echo 'hnet/ already exists, skipping clone'
fi

# Our FD-DRAT code
# Option A: if uploaded as Kaggle Dataset, it's already mounted
# Option B: clone from your private repo:
# git clone https://github.com/YOUR_USERNAME/laststand-code.git fddrat
# For now assume the code is in /kaggle/input/fddrat-code/
if [ -d /kaggle/input/fddrat-code ]; then
    cp -r /kaggle/input/fddrat-code /kaggle/working/code
    echo 'FD-DRAT code copied from Kaggle Dataset'
else
    echo 'WARNING: FD-DRAT code not found. Upload the repo as a Kaggle Dataset named fddrat-code.'
fi

## Cell 3 — Install dependencies

In [ ]:
%%bash
set -e
cd /kaggle/working

# Install OAT package and all its dependencies
pip install -e ./oat --quiet

# Install remaining project deps
pip install hydra-core>=1.3.2 pytorch-lightning>=2.6.1 wandb>=0.25.1 --quiet

echo 'Dependencies installed'

## Cell 4 — Download LIBERO10 dataset

In [ ]:
%%bash
set -e
mkdir -p /kaggle/working/data/libero

ZARR_PATH='/kaggle/working/data/libero/libero10_N500.zarr'

if [ -d "$ZARR_PATH" ]; then
    echo 'Dataset already downloaded'
else
    pip install huggingface-hub --quiet
    # Download the zarr zip and extract
    python - <<'EOF'
from huggingface_hub import hf_hub_download
import zipfile, os

zip_path = hf_hub_download(
    repo_id='chaoqi-liu/libero10_N500.zarr',
    filename='libero10_N500.zarr.zip',
    repo_type='dataset',
    local_dir='/kaggle/working/data/libero'
)
print(f'Downloaded to {zip_path}')
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall('/kaggle/working/data/libero/')
os.remove(zip_path)
print('Extracted and cleaned up zip')
EOF
fi

ls /kaggle/working/data/libero/

## Cell 5 — Train OAT Tokenizer

Saves checkpoint to `oat/output/{date}/{time}_train_oattok_libero10_N500/checkpoints/latest.ckpt`.

Expected time: ~2-3h on T4 for 300 epochs (enough for good reconstruction).
Full 5001 epochs from the paper takes ~12h.

In [ ]:
%%bash
set -e
cd /kaggle/working/oat

HYDRA_FULL_ERROR=1 python oat/workspace/train_oattok.py \
    --config-path /kaggle/working/oat/oat/config \
    --config-name train_oattok \
    task/tokenizer=libero/libero10 \
    task.tokenizer.dataset.zarr_path=/kaggle/working/data/libero/libero10_N500.zarr \
    training.num_epochs=300 \
    training.checkpoint_every=50 \
    logging.mode=online \
    logging.project=VLA-experiment

## Cell 5b — Find tokenizer checkpoint path

In [ ]:
import glob
import os

ckpts = glob.glob('/kaggle/working/oat/output/**/checkpoints/latest.ckpt', recursive=True)
if not ckpts:
    raise RuntimeError('No tokenizer checkpoint found! Check Cell 5 output.')

# Take the most recently modified one
TOK_CKPT = max(ckpts, key=os.path.getmtime)
print(f'Using tokenizer checkpoint: {TOK_CKPT}')

## Cell 6 — Train FD-DRAT Policy

In [ ]:
import subprocess, sys

FDDRAT_CODE = '/kaggle/working/code'
TOK_CKPT    # set in Cell 5b

cmd = [
    sys.executable, f'{FDDRAT_CODE}/run.py',
    'strategy=single_gpu',
    f'model.tokenizer_ckpt={TOK_CKPT}',
    f'dataset_path=/kaggle/working/data/libero/libero10_N500.zarr',
    'batch_size=16',
    'learning_rate=3e-4',
]

env = os.environ.copy()
env['PYTHONPATH'] = f"{FDDRAT_CODE}:{os.environ.get('PYTHONPATH', '')}"

result = subprocess.run(cmd, cwd=FDDRAT_CODE, env=env)
if result.returncode != 0:
    raise RuntimeError(f'Training failed with code {result.returncode}')

## Cell 7 — Save outputs

In [ ]:
import shutil, glob

# Save FD-DRAT checkpoints
fddrat_ckpts = glob.glob('/kaggle/working/code/outputs/**/checkpoints/*.ckpt', recursive=True)
print('FD-DRAT checkpoints:')
for c in sorted(fddrat_ckpts):
    print(' ', c)

# Copy to /kaggle/working/ so they appear in session output
os.makedirs('/kaggle/working/fddrat_checkpoints', exist_ok=True)
for c in fddrat_ckpts:
    shutil.copy(c, '/kaggle/working/fddrat_checkpoints/')
print('Saved to /kaggle/working/fddrat_checkpoints/')